# How to Program Your Own Semi-Discrete Optimal Transport at Home (2D)

## Prerequisites

Like previous tutorials, you need to install the dependencies with
```sh
pip install -r requirements.txt
```

as well as `geogram`'s python bindings with
```sh
pip install geogram
```

We can then import all required modules:


In [ ]:
import numpy as np
np.random.seed(0)

import scipy
import geogram as geo

## Step 1: Laguerre diagram



### Create a square

The first thing we need to do is defining the domain. 

We create a unit square, using the `shape.quad()` method of `geogram`'s `shape` module. 

In [ ]:
(domain_vertices, domain_triangles) = geo.shape.quad()

### Sample it with random points

Now we create a point set. In this example, we will use a set of N=100 points picked randomly in our unit square:

In [ ]:
N = 100
INITIAL_SEEDS = np.random.uniform(size=(N, 2))

### Compute a Laguerre diagram

And finally, we can compute the Laguerre diagram (restricted to the domain). 

Laguerre diagrams are defined from a pointset and a vector of "weights". 
Here we use (for now) a vector of weights with all weights set to 0.

`geogram` can directly compute a Laguerre diagram from a set of points (let's call them seeds), their (optional) weights and a domain mesh.

In [ ]:
weights  = np.zeros(N)
laguerre = geo.Voronoi(INITIAL_SEEDS, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)

The diagram can be directly rendered thanks to `geogram`'s data structure.

Let's add a nice method for the next experiments:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as tri


def plot_voronoi(voronoi, ax=None):
    if ax == None:
        _, ax = plt.subplots()

    ax.set_aspect('equal')
    triang  = tri.Triangulation(voronoi.q[:, 0], voronoi.q[:, 1], voronoi.t)
    ax.tripcolor(triang, voronoi.tseed, shading='flat')
    ax.set_axis_off()


plot_voronoi(laguerre)

## Step 2: Change the weight of a cell in a Laguerre diagram

Let us now see what happens when we change the weight of one cell in the Laguerre diagram. 

In [ ]:
_, axs = plt.subplots(1, 2)

weights  = np.zeros(N)
laguerre = geo.Voronoi(INITIAL_SEEDS, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)

plot_voronoi(laguerre, axs[0])

weights[0] += 0.08 # Try with higer values!
laguerre = geo.Voronoi(INITIAL_SEEDS, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)

plot_voronoi(laguerre, axs[1])

Notice that slightly increasing the weight of a point results in a bigger cell.

## Step 3: Translating a Laguerre diagram by just changing the weight

Now you may think about a Laguerre diagram as a Voronoi diagram plus tuning buttons (the weights). By changing the tuning buttons, one may increase or decrease the size of the associated Laguerre cells. In fact there is more: did you know that you can translate a Laguerre diagram by an arbitrary vector just by changing the weights?

Let us take a close look at the common boundary between two cells $i$ and $j$, associated with points $x_i$ and $x_j$. A point $x$ that is on this common boundary satisfies the following equation:

$$d^2\left( x, x_i \right) − w_i = d^2\left( x, x_j \right) − w_j$$

where $w_i$ and $w_j$ are the weights associated with points $x_i$ and $x_j$ and where $d^2(.,.)$ denotes the squared distance between two points.

Side note: One can notice that this equation only involves linear terms in the coordinates of $x$ (the squared lengh of $x$ appears on both sides of the equation and cancel out), hence this common boundary is a straight line.

Suppose that $w_i = w_j = 0$, then what you have is the Voronoi diagram of $x_i$ and $x_j$. Imagine now that you want to translate the whole diagram by an arbitrary 2D vector $T.$ Is is possible to do so just by changing $w_i$ and $w_j$? Try to derive it on your own before reading further.

In other words, how can we set $w_i$ and $w_j$ in such a way that the set of points $x$ that satisfy: $$d^2\left(x, x_i + T\right) − w_i = d^2\left(x, x_j + T \right) − w_j$$ corresponds to the same straight line as the edge between the Voroonoi cells of $x_i$ and $x_j$?

It can be easily checked that setting $w_i = -2(T \cdot x_i)$ and $w_j = -2(T \cdot x_j)$ works (where $\cdot$ denotes the dot product).

OK, let us now program it!

First thing we need to do is to group a little bit our points in the lower-left corner of the square, so that we will still see them when we will translate them. 

Right after we create the point set, modify the coordinates:

In [ ]:
seeds = INITIAL_SEEDS / 2.

We can now apply our translation and compare it with the original result:

In [ ]:
_, axs = plt.subplots(1, 2)

# Original
laguerre = geo.Voronoi(seeds, np.zeros(N), domain_vertices=domain_vertices, domain_simplices=domain_triangles)
plot_voronoi(laguerre, axs[0])

# Translated
T = np.array([0.25, 0.]) # Try to increase this value!

translation = np.repeat(T[np.newaxis, :], repeats=N, axis=0) # Per seed translation
w           = np.sum(translation * seeds, axis=1)            # Per seed dot product

weights  = -2. * w
laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
plot_voronoi(laguerre, axs[1])

What we have learnt so far:

- how to compute a Laguerre diagram with geogram in Python
- by changing the weights of a Laguerre diagram, one can change the area of the cells
- by changing the weights of a Laguerre diagram, one can translate it at a arbitrary position

## Step 4: Semi-discrete optimal transport without step-length control

Semi-discrete Optimal Transport gives a way of computing the weights in such a way that **all the cells have prescribed areas**. As before, we will start from a random distribution of points, and from their Voronoi diagram (all weights set to zero). Then we will iteratively update the weights until all the Laguerre cells have the prescribed areas (it will be the same, $\nu_i=\frac{1}{N'}$, for all cell iin the present example).

We know (detailed derivation here) that this means solving a discrete Poisson system. 

To assemble the Poisson system, we first need some helper functions:

In [ ]:
def triangle_area(vertices, triangles):
    """
    @brief Computes the area of a mesh triangle
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] triangles an array with the three vertices indices of the triangle
    @details Works also when T is an array of triangles (then it returns
        the array of triangle areas). This is why the ellipsis (...)
        is used (here it means indexing/slicing through the last dimension)
    """
    v1 = triangles[..., 0]
    v2 = triangles[..., 1]
    v3 = triangles[..., 2]

    u = vertices[v2] - vertices[v1]
    v = vertices[v3] - vertices[v1]
    return np.abs(0.5 * (u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]))


def distance(vertices, v1, v2):
    """
    @brief Computes the length of a mesh edge
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] v1, v2 the mesh extremities indices.
    @details v1 and v2 can be also arrays (then returns the array of distances).
    """
    axis = v1.ndim if hasattr(v1, 'ndim') else 0
    return np.linalg.norm(vertices[v2] - vertices[v1], axis=axis)


def hessian(voronoi):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """
    NO_INDEX = -1  # Special value for invalid indices (edge on border)

    # Compute one entry per triangle half-edge (3*nt entries) with:
    # I is the seed associated with the triangle
    # J is the seed on the other side of the triangle's edge (NO_INDEX on border)
    # V1 and V2 are the two vertices of the triangle
    i = voronoi.tseed

    tadj = voronoi.tadj
    tadj = tadj[:, [1, 2, 0]] # <- permute columns to match std convention for triangulations:
                              #   different in geogram meshes because they also support n-sided polygons
                                      
    j = tadj.T.flatten().astype(np.int32)
    j = np.where(j != NO_INDEX, i[j], NO_INDEX).astype(np.int32)  # lookup seed on other side
    
    i  = np.concatenate((i, i, i))
    v1 = np.concatenate((voronoi.t[:, 1], voronoi.t[:, 2], voronoi.t[:, 0]))
    v2 = np.concatenate((voronoi.t[:, 2], voronoi.t[:, 0], voronoi.t[:, 1]))

    # Remove (i,j,v1,v2) index quadruplets that correspond to
    #   - border triangle edges (j == NO_INDEX)
    #   - triangle edges inside Laguerre cell (i == j)
    qidx = np.column_stack((i, j, v1, v2))
    qidx = qidx[np.logical_and(i != j, j != NO_INDEX)]

    i = qidx[:, 0]  # re-extract i, j, v1, v2
    j = qidx[:, 1]
    v1 = qidx[:, 2]
    v2 = qidx[:, 3]

    # Now we can compute the vector of coefficient (note: v1, v2, i, j are vectors)
    val = -distance(voronoi.q, v1, v2) / (2. * distance(voronoi.seeds, i, j))

    return i, j, val


def measures(voronoi):
    """
    @brief Computes the measures of the Laguerre cells
    @return the vector of Laguerre cells measures
    @details Uses the current Laguerre diagram (in self.Laguerre)
    """
    # See comments about XY,T,trgl_seed,nt in compute_Laguerre_diagram()
    measures = np.zeros(len(voronoi.seeds))
    np.add.at(measures, voronoi.tseed, triangle_area(voronoi.q, voronoi.t))
    return measures

Let $F$, the discrete dual Kantorovich function. The Poisson problem that we want to solve is composed of $-\nabla F$ and $\nabla^2 F$, $F$'s gradient and hessian matrix (accross the implementation, a reader may notice that it is in fact equals to the P1 Laplacian).

You can find detailed information on these derivation in [Notions of optimal transport theory and how to implement them on a computer](https://hal.science/hal-01717967v1/file/OT-preprint.pdf).

For simplification purposes, we refer to $\nabla^2F$ as $H$ and $-\nabla F$ as $b$.

We can create the sparse matrix $H$ and initialize the right-hand side b of our Poisson system. Each component of b is supposed to be equal to the target area ($\nu_i=\frac{1}{N}$ in our case) minus the current area of the Laguerre cell. Hence we update b as follows:

In [ ]:
def newton_step(seeds, weights, alpha=1./8., regularization=1e-6):
    nu_i     = 1. / len(seeds) # Target areas
    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)

    # Create and compute H
    i, j, val = hessian(laguerre)

    n = len(seeds)
    diag = np.zeros(n, np.float64) # Diagonal (initialized to zero)
    np.add.at(diag, i, -val)       # =minus sum extra-diagonal coefficients
    if regularization != 0.0:
        diag += regularization * nu_i

    H = scipy.sparse.csr_array((val, (i, j)), shape=(n, n))
    s = np.arange(n, dtype=np.int32)
    H += scipy.sparse.csr_array((diag, (s, s)), shape=(n, n))
    
    # rhs (minus gradient of Kantorovich dual) = desired areas - actual areas
    b = nu_i - measures(laguerre)
    if regularization != 0.0:
        b -= regularization * nu_i * weights

    p = scipy.sparse.linalg.spsolve(H, b)
    return weights + alpha * p

In [ ]:
seeds = INITIAL_SEEDS / 1.25 # shift seeds s.t. cells have varying areas

In [ ]:
_, axs = plt.subplots(2, 4)

# Original
laguerre = geo.Voronoi(seeds, np.zeros(N), domain_vertices=domain_vertices, domain_simplices=domain_triangles)
plot_voronoi(laguerre, axs[0][0])

weights  = np.zeros(N)
for i in range(1, 16):
    weights = newton_step(seeds, weights)
    if i % 2 != 0:
        continue

    ii = i // 2

    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
    plot_voronoi(laguerre, axs[ii // 4][ii % 4])

We notice that our example converges to a Laguerre diagram with the same area for all the cells.

To make things more interesting, let's divide point coordinates by 10 (instead of 2):

In [ ]:
seeds = INITIAL_SEEDS / 10.

In [ ]:
_, axs = plt.subplots(4, 4)

# Original
laguerre = geo.Voronoi(seeds, np.zeros(N), domain_vertices=domain_vertices, domain_simplices=domain_triangles)
plot_voronoi(laguerre, axs[0][0])

weights  = np.zeros(N)
for i in range(1, 256):
    weights = newton_step(seeds, weights, alpha=1./8.) # Try to change alpha value
    if i % 16 != 0:
        continue

    ii = i // 16

    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
    plot_voronoi(laguerre, axs[ii // 4][ii % 4])

As you can see, **it no longer converges**. Can you find a quick-and-dirty fix for this example?

OK, let us try with a smaller alpha, let us say alpha=1.0/10.0? 

No, does not converge, and what about alpha=1.0/100.0? 

Yes, it does the right thing. Now, it would not be reasonable to let the user tune alpha, so let us move to the next step!

## Step 5: step-length control (Kitagawa-Merigot-Thibert)

There is a [nice article](https://arxiv.org/abs/1603.05579) by Jun Kitagawa, Quentin Merigot and Boris Thibert, with a method (called "KMT" after its authors) to choose alpha in such a way that the convergence of the Newton method is guaranteed.

The idea is to start with alpha=1, then halve alpha until certain conditions are satisfied. These conditions says that:

- all Laguerre cells should be larger than half the smallest Voronoi cell
- all Laguerre cells should be larger than half the smallest target area (0.5/N in our case)
- the norm of g after the update should be smaller than (1 - 0.5*alpha)*norm(g) before the update, where g is the vector of the differences between the target areas and the actual areas of the Laguerre cells ($g_i = nu_i - Area(Lag(i)) = 1/N - Area(Lag(i))$)

Remark the first condition forbids to have any empty Laguerre cell in the diagram.

Now we can update our function to take into account the three KMT conditions. For the first two ones, we need to compute the area of the smallest Voronoi cell. Since the first iteration computes a Voronoi diagram (all weights set to zero), we compute it right after:

In [ ]:
def newton_step(seeds, weights, regularization=1e-6, verbose=False):
    # Compute smallest area threshold
    voronoi = geo.Voronoi(seeds, np.zeros(len(seeds)), domain_vertices=domain_vertices, domain_simplices=domain_triangles)

    smallest_cell_area      = np.min(measures(voronoi))
    smallest_cell_threshold = 0.5 * min(smallest_cell_area, 1.0/N)

    # Target area
    n    = len(seeds)
    nu_i = 1. / n

    # Current Laguerre diagram (using weights)
    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
    
    # Create and compute H with scipy
    i, j, val = hessian(laguerre)

    diag = np.zeros(n, np.float64)  # Diagonal (initialized to zero)
    np.add.at(diag, i, -val)  # =minus sum extra-diagonal coefficients
    if regularization != 0.0:
        diag += regularization * nu_i

    H  = scipy.sparse.csr_array((val, (i, j)), shape=(n, n))
    s  = np.arange(n, dtype=np.int32)
    H += scipy.sparse.csr_array((diag, (s, s)), shape=(n, n))
    
    # rhs (minus gradient of Kantorovich dual) = desired areas - actual areas
    b = nu_i - measures(laguerre)
    if regularization != 0.0:
        b -= regularization * nu_i * weights

    p = scipy.sparse.linalg.spsolve(H, b)


    # Compute alpha with KMT
    alpha   = 1.0 
    weights = weights + alpha * p  # Start with Newton step
    g_norm  = np.linalg.norm(b)    # norm of gradient at current step (KMT #2)
    for _ in range(100):
        laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)

        # g (grad of Kantorovich dual) at substep = actual areas - desired areas
        g = measures(laguerre)
        smallest_area = np.min(g) # for KMT criterion 1
        g -= nu_i

        # Check KMT criteria #1 (cell area) and #2 (gradient norm)
        g_norm_k = np.linalg.norm(g)
        kmt_1    = (smallest_area > smallest_cell_threshold)  # criterion 1: cell area
        kmt_2    = (g_norm_k <= (1.0 - 0.5 * alpha) * g_norm) # criterion 2: gradient norm

        if verbose:
            print(f' KMT #1 (area): {kmt_1} {smallest_area}>{smallest_cell_threshold}')
            print(f' KMT #2 (grad): {kmt_2} {g_norm_k}<={(1.0 - 0.5 * alpha) * g_norm}')
            
        if kmt_1 and kmt_2:
            break

        alpha    = alpha / 2.0
        weights -= alpha * p

    return weights

In [ ]:
_, axs = plt.subplots(4, 4)

laguerre = geo.Voronoi(seeds, np.zeros(N), domain_vertices=domain_vertices, domain_simplices=domain_triangles)
plot_voronoi(laguerre, axs[0][0])

weights = np.zeros(N)
for i in range(1, 16):
    weights = newton_step(seeds, weights) # Try to change alpha value

    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
    plot_voronoi(laguerre, axs[i // 4][i % 4])

Instead of trying to find a good fixed number of iteration, we can use an error threshold to automatically detect when the optimization converged.

In [ ]:
import datetime
starttime = datetime.datetime.now()

seeds   = INITIAL_SEEDS / 10.
weights = np.zeros(N)

nu_i      = (1. / N)
threshold = nu_i * 0.001  # .1% of desired cell area
current_error = np.inf

iteration = 0
plots     = []
while current_error > threshold:
    weights = newton_step(seeds, weights) # Try to change alpha value

    iteration += 1
    print('Step: {:2d}\tWorst cell area error: {:.6f}'.format(iteration, 100.0 * current_error / nu_i))

    laguerre = geo.Voronoi(seeds, weights, domain_vertices=domain_vertices, domain_simplices=domain_triangles)
    current_error = np.max(np.abs(nu_i - measures(laguerre)))

    triang  = tri.Triangulation(laguerre.q[:, 0], laguerre.q[:, 1], laguerre.t)
    plots += [ ( triang, laguerre.tseed ) ]

print('-'*10)
print(f'Total elapsed time for OT: {datetime.datetime.now() - starttime}')
print(f'Converged in {iteration} iterations')

In [ ]:
fig, axes = plt.subplots( 2, 4 )

indices = np.round(np.linspace(0, iteration-1, 8)).astype(int)
for n, i in enumerate(indices):
    triangulation, tseed = plots[i]

    axes[n // 4, n % 4].set_aspect('equal')
    axes[n // 4, n % 4].tripcolor(triangulation, tseed, shading='flat')
    axes[n // 4, n % 4].set_title(f'Step {i}')
    axes[n // 4, n % 4].set_axis_off()

fig.tight_layout()